In [ ]:
from qick.averager_program import AveragerProgram
from qcodes import Instrument, Parameter, Station
from qickodes import QickInstrument
# Base de datos
from qcodes.dataset import (Measurement, load_or_create_experiment, initialise_or_create_database_at, experiments, load_by_run_spec)

In [ ]:
class ToneProgram(AveragerProgram):
    def initialize(self):
        cfg = self.cfg

        # --- declare the DAC channel: bookkeeping only, no ASM emitted yet ---
        self.declare_gen(ch=cfg["gen_ch"], nqz=cfg["nqz"])

        # --- convert physical units (MHz) into register values ---
        freq_reg = self.freq2reg(cfg["freq_MHz"], gen_ch=cfg["gen_ch"])
        length_cycles = self.us2cycles(cfg["length_us"], gen_ch=cfg["gen_ch"])

        # --- write the pulse parameters into registers (this DOES touch prog_list) ---
        self.set_pulse_registers(
            ch=cfg["gen_ch"],
            style=cfg["style"],        # "const", "arb", or "flat_top" -- shape of the pulse
            freq=freq_reg,
            phase=0,
            gain=cfg["gain"],          # integer DAC code, up to self.get_maxv(cfg["gen_ch"])
            length=length_cycles,
            mode=cfg.get("mode", "oneshot"),  # "oneshot" or "periodic" -- THIS is what makes it a CW tone
        )

        self.synci(200)  # let the DAC/mixer settings settle before the first pulse

    def body(self):
        self.pulse(ch=self.cfg["gen_ch"])   # actually emit the pulse (schedules it)
        self.wait_all()                     # wait for all pending pulses/readouts to finish
        self.sync_all(self.us2cycles(1))    # pad 1 us before the next rep

In [ ]:
# --- connect to the board over Pyro4 (fill in your actual nameserver IP; port must match pyro_service.py's ns_port) ---
# If re-running this cell, close the old instance first: qick_instrument.close()
qick_instrument = QickInstrument("ip.address.of.board", ns_port=8000, name="qick")
print(qick_instrument.soccfg)

In [ ]:
config = {
    "gen_ch": 0,          # which DAC channel to use
    "nqz": 1,            # DAC resolution in bits
    "freq_MHz": 100,      # frequency of the tone in MHz
    "gain": 0,            # DAC gain (integer code)
    "length_us": 1000,    # length of the pulse in microseconds
    "style": "const",     # pulse shape: "const", "arb", or "flat_top"
    "mode": "periodic",    # pulse mode: "oneshot" or "periodic"
}

prog = ToneProgram(qick_instrument.soccfg, config)

In [ ]:
prog.run_rounds(qick_instrument.soc)  # run the program for 1 round (1 pulse)

In [ ]:
# --- lock the board's sampling clocks to the spectrum analyzer's 10 MHz reference ---
# Requires the analyzer's REF OUT to already be cabled into the board's CLK_IN SMA.
# This re-downloads the bitstream, so do it once here, right after connecting --
# not mid-experiment, since it wipes any previously-loaded tProc program state.
qick_instrument.soc.config_clocks(force_init_clks=True, clk_output=False, external_clk=True)
print("clocks locked:", qick_instrument.soc.clocks_locked())

In [ ]:
class DacCharacterization(Instrument):
    def __init__(self, name, qick_instrument, gen_ch, freq_MHz=1000, style="const",
                 mode="periodic", nqz=1, length_us=10):
        super().__init__(name)
        self.qick_instrument = qick_instrument
        self.gen_ch = gen_ch
        self.nqz = nqz
        self.freq_MHz = freq_MHz
        self.style = style
        self.mode = mode            # "periodic" = true CW output, needed to read on a spectrum analyzer
        self.length_us = length_us
        self._gain = None           # last gain actually played, so _set_freq can replay at the new freq

        self.add_parameter(
            "gain",
            label="DAC gain",
            unit="DAC code",
            set_cmd=self._set_gain,
            get_cmd=lambda: self._gain,          # just caches the last value set
        )

        self.add_parameter(
            'freq_MHz',
            label='Frequency',
            unit='MHz',
            set_cmd=self._set_freq,
            get_cmd=lambda: self.freq_MHz,
        )

    def _set_freq(self, freq):
        self.freq_MHz = freq
        if self._gain is not None:
            # a tone is already playing -- replay it at the new frequency, same gain
            self._play_tone()
            
    def _set_gain(self, gain):
        self._gain = gain
        self._play_tone()

    def make_program(self):
        return ToneProgram(self.qick_instrument.soccfg,
        {
            "gen_ch": self.gen_ch, "freq_MHz": self.freq_MHz,
            "gain": int(self._gain), "length_us": self.length_us, "reps": 1,
            "style": self.style, "mode": self.mode, "nqz": self.nqz,
        })

    def _play_tone(self):
        prog = self.make_program()
        prog.run_rounds(self.qick_instrument.soc)

In [ ]:
dac_char = DacCharacterization("dac_char", qick_instrument, gen_ch=0)

In [ ]:
# --- database + experiment setup (was imported but never called) ---
initialise_or_create_database_at("./database/dac_characterize.db")
experiment = load_or_create_experiment("dac_characterize", sample_name="none")

station = Station()
station.add_component(qick_instrument)
station.add_component(dac_char)

# the spectrum analyzer isn't a QCoDeS instrument here, so this is a plain Parameter
# you fill in by hand -- see the "no readout on the RFSoC" discussion
power_dbm = Parameter("power_dbm", label="Measured power", unit="dBm")

meas = Measurement(exp=experiment, station=station, name="dac_gain_characterization")
meas.register_parameter(dac_char.gain)
meas.register_parameter(power_dbm, setpoints=[dac_char.gain])   # gain is the x-axis

gains = [1000, 5000, 10000, 15000, 20000, 25000, 30000]


In [ ]:
with meas.run() as datasaver:
    for g in gains:
        dac_char.gain.set(g)                                    # plays the tone
        measured = float(input(f"gain={g}: read spectrum analyzer, enter dBm: "))
        datasaver.add_result((dac_char.gain, g), (power_dbm, measured))

dataset = datasaver.dataset
print(f"run_id = {dataset.run_id}")